# Solutions — Dictionaries: Lookup, Aggregation, Merging, Flattening

Exercise notebook: `02_collections/04_dictionaries.ipynb`. Each solution has: working code, an explanation, complexity, common mistakes, and a Java-style vs Pythonic comparison where it teaches something.

## Exercise 1 — 🟢 Easy · Build a lookup index

**Working solution**

In [ ]:
def index_by(records, key):
    index = {}
    for record in records:
        value = record[key]
        if value in index:
            raise ValueError(f"duplicate {key}: {value!r}")
        index[value] = record
    return index

**Explanation**

An index turns repeated O(n) searches into O(1) lookups — build it once before a join-like loop.

**Complexity:** O(n).

**Common mistakes**

- `{r[key]: r for r in records}` silently keeps the last duplicate — fine only if that's the intent.

## Exercise 2 — 🟢 Easy / 🟡 Medium · Invert a one-to-many mapping

**Working solution**

In [ ]:
from collections import defaultdict

def invert(mapping):
    inverted = defaultdict(set)
    for group, members in mapping.items():
        for member in members:
            inverted[member].add(group)
    return {member: sorted(groups) for member, groups in sorted(inverted.items())}

**Explanation**

`defaultdict(set)` both groups and dedupes. The final dict comprehension sorts keys and values in one expression.

**Complexity:** O(n log n).

**Common mistakes**

- `dict.setdefault(member, []).append(group)` — works but keeps duplicates.

## Exercise 3 — 🟡 Medium · Revenue by any dimension

**Working solution**

In [ ]:
from collections import defaultdict

def revenue_by(orders, field):
    totals = defaultdict(float)
    for order in orders:
        try:
            qty, price = int(order["quantity"]), float(order["unit_price"])
        except (KeyError, ValueError, TypeError):
            continue
        if qty <= 0 or price < 0:
            continue
        totals[order[field]] += qty * price
    return dict(sorted(((k, round(v, 2)) for k, v in totals.items()), key=lambda kv: kv[1], reverse=True))

**Explanation**

Group-by in plain Python = `defaultdict` accumulator + a final sort of `.items()`. `dict(sorted(...))` keeps the sorted order.

**Complexity:** O(n + k log k).

**Common mistakes**

- Sorting a dict and expecting the dict itself to be 'sorted' afterwards — sort its items into a new dict.

## Exercise 4 — 🟠 Hard · Deep-merge layered configuration

**Working solution**

In [ ]:
import copy

def deep_merge(base, override):
    result = copy.deepcopy(base)
    for key, value in override.items():
        if value is None:
            result.pop(key, None)
        elif isinstance(value, dict) and isinstance(result.get(key), dict):
            result[key] = deep_merge(result[key], value)
        else:
            result[key] = copy.deepcopy(value)
    return result

**Explanation**

`{**a, **b}` and `a | b` only merge the top level. Recursion handles nesting; `deepcopy` guarantees the result shares
no mutable objects with the inputs (otherwise editing the merged config would edit your defaults).

**Complexity:** O(total keys).

**Common mistakes**

- `base.update(override)` — mutates base and replaces whole nested sections.
- Shallow copy → shared nested lists.

## Challenge — 🔴 Very Hard · Flatten and unflatten JSON documents

**Working solution**

In [ ]:
def flatten_dict(doc, sep="."):
    flat = {}

    def walk(node, prefix):
        if isinstance(node, dict) and node:
            items = node.items()
        elif isinstance(node, list) and node:
            items = ((str(i), v) for i, v in enumerate(node))
        else:
            flat[prefix] = node
            return
        for key, value in items:
            walk(value, f"{prefix}{sep}{key}" if prefix else str(key))

    for key, value in doc.items():
        walk(value, str(key))
    return flat


def _listify(node):
    if not isinstance(node, dict) or not node:
        return node
    converted = {k: _listify(v) for k, v in node.items()}
    if all(k.isdigit() for k in converted) and sorted(map(int, converted)) == list(range(len(converted))):
        return [converted[str(i)] for i in range(len(converted))]
    return converted


def unflatten_dict(flat, sep="."):
    root = {}
    for path, value in flat.items():
        *parents, leaf = path.split(sep)
        node = root
        for part in parents:
            node = node.setdefault(part, {})
        node[leaf] = value
    return {k: _listify(v) for k, v in root.items()}

**Explanation**

Flattening is a depth-first walk carrying the path prefix. Unflattening builds a dict tree with `setdefault`, then a
second pass converts digit-keyed dicts back to lists. `*parents, leaf = path.split(sep)` is extended unpacking — the
last element and "everything before it" in one line. This is exactly what Spark's `select("a.b")` / Glue's `Relationalize` do.

**Complexity:** O(total leaves × depth).

**Common mistakes**

- Dropping empty lists/dicts so the round trip loses data.
- Converting a level to a list when keys are `0, 2` (non-consecutive).